In [1]:
import pandas as pd
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI

In [6]:
df_temp = pd.read_csv("https://datahub.io/core/global-temp/r/0.csv")
df_temp.head()

,Source,Year,Mean
0,GCAG,1850,-0.4265
1,GCAG,1851,-0.2635
2,GCAG,1852,-0.2241
3,GCAG,1853,-0.2623
4,GCAG,1854,-0.2936


In [7]:
chat = ChatOpenAI(model="gpt-3.5-turbo-0125")

agent_temp = create_pandas_dataframe_agent(
    chat,
    df_temp,
    verbose=True,
    agent_type="tool-calling"
)

In [8]:
# 1 - Qual é a temperatura média global do último ano?
resposta = agent_temp.invoke({"input": "Qual é a temperatura média global do último ano?"})
print(resposta)



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df[df['Year'] == df['Year'].max()]['Mean'].values[0]"}`


1.0323A temperatura média global do último ano é de 1.0323.

> Finished chain.
{'input': 'Qual é a temperatura média global do último ano?', 'output': 'A temperatura média global do último ano é de 1.0323.'}


In [10]:
# 2 - Qual foi o ano com a maior temperatura média?
resposta = agent_temp.invoke({"input": "Qual foi o ano com a maior temperatura média?"})
print(resposta)



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df[df['Mean'] == df['Mean'].max()]['Year'].values[0]"}`


2024O ano com a maior temperatura média registrada no dataframe foi 2024.

> Finished chain.
{'input': 'Qual foi o ano com a maior temperatura média?', 'output': 'O ano com a maior temperatura média registrada no dataframe foi 2024.'}


In [11]:
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_openai import ChatOpenAI

In [12]:
db_sakila = SQLDatabase.from_uri("sqlite:///files/sakila_master.db")

chat = ChatOpenAI(model="gpt-3.5-turbo-0125")

agente_executor_sakila = create_sql_agent(
    chat,
    db=db_sakila,
    agent_type="tool-calling",
    verbose=True
) 

In [13]:
response = agente_executor_sakila.invoke({"input": "Quais são os filmes mais alugados?"})
print(response)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


actor, address, category, city, country, customer, film, film_actor, film_category, film_text, inventory, language, payment, rental, staff, store
Invoking: `sql_db_schema` with `{'table_names': 'film'}`



CREATE TABLE film (
	film_id INTEGER NOT NULL, 
	title VARCHAR(255) NOT NULL, 
	description TEXT DEFAULT NULL, 
	release_year VARCHAR(4) DEFAULT NULL, 
	language_id SMALLINT NOT NULL, 
	original_language_id SMALLINT DEFAULT NULL, 
	rental_duration SMALLINT DEFAULT 3 NOT NULL, 
	rental_rate DECIMAL(4, 2) DEFAULT (4.99) NOT NULL, 
	length SMALLINT DEFAULT NULL, 
	replacement_cost DECIMAL(5, 2) DEFAULT (19.99) NOT NULL, 
	rating VARCHAR(10) DEFAULT 'G', 
	special_features VARCHAR(100) DEFAULT NULL, 
	last_update TIMESTAMP NOT NULL, 
	PRIMARY KEY (film_id), 
	CONSTRAINT fk_film_language FOREIGN KEY(language_id) REFERENCES language (language_id), 
	CONSTRAINT fk_film_language_original FOREIGN KEY(orig

/Users/luizfelipew/Documents/git/AI/Exploring-LLMs/agents/agentes-ai-apps/.venv/lib/python3.13/site-packages/langchain_community/utilities/sql_database.py:313: SAWarning: Cannot correctly sort tables; there are unresolvable cycles between tables "staff, store", which is usually caused by mutually dependent foreign key constraints.  Foreign key constraints involving these tables will not be considered; this warning may raise an error in a future release.
  metadata_table_names = [tbl.name for tbl in self._metadata.sorted_tables]
/Users/luizfelipew/Documents/git/AI/Exploring-LLMs/agents/agentes-ai-apps/.venv/lib/python3.13/site-packages/langchain_community/utilities/sql_database.py:325: SAWarning: Cannot correctly sort tables; there are unresolvable cycles between tables "staff, store", which is usually caused by mutually dependent foreign key constraints.  Foreign key constraints involving these tables will not be considered; this warning may raise an error in a future release.
  for tb


Invoking: `sql_db_query` with `{'query': 'SELECT title, COUNT(*) AS total_rentals FROM film JOIN inventory ON film.film_id = inventory.film_id JOIN rental ON inventory.inventory_id = rental.inventory_id GROUP BY film.film_id ORDER BY total_rentals DESC LIMIT 10'}`


[('BUCKET BROTHERHOOD', 34), ('ROCKETEER MOTHER', 33), ('SCALAWAG DUCK', 32), ('RIDGEMONT SUBMARINE', 32), ('JUGGLER HARDLY', 32), ('GRIT CLOCKWORK', 32), ('FORWARD TEMPLE', 32), ('ZORRO ARK', 31), ('WIFE TURN', 31), ('TIMBERLAND SKY', 31)]Os filmes mais alugados são:
1. BUCKET BROTHERHOOD - 34 aluguéis
2. ROCKETEER MOTHER - 33 aluguéis
3. SCALAWAG DUCK - 32 aluguéis
4. RIDGEMONT SUBMARINE - 32 aluguéis
5. JUGGLER HARDLY - 32 aluguéis
6. GRIT CLOCKWORK - 32 aluguéis
7. FORWARD TEMPLE - 32 aluguéis
8. ZORRO ARK - 31 aluguéis
9. WIFE TURN - 31 aluguéis
10. TIMBERLAND SKY - 31 aluguéis

> Finished chain.
{'input': 'Quais são os filmes mais alugados?', 'output': 'Os filmes mais alugados são:\n1. BUCKET BROTHERHOOD - 34 aluguéi

In [14]:
response = agente_executor_sakila.invoke({"input": "Quais atores atuaram em mais filmes?"})
print(response)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


actor, address, category, city, country, customer, film, film_actor, film_category, film_text, inventory, language, payment, rental, staff, store
Invoking: `sql_db_schema` with `{'table_names': 'actor, film_actor'}`



CREATE TABLE actor (
	actor_id NUMERIC NOT NULL, 
	first_name VARCHAR(45) NOT NULL, 
	last_name VARCHAR(45) NOT NULL, 
	last_update TIMESTAMP NOT NULL, 
	PRIMARY KEY (actor_id)
)

/*
3 rows from actor table:
actor_id	first_name	last_name	last_update
1.0000000000	PENELOPE	GUINESS	2020-12-23 07:12:29
2.0000000000	NICK	WAHLBERG	2020-12-23 07:12:29
3.0000000000	ED	CHASE	2020-12-23 07:12:29
*/


CREATE TABLE film_actor (
	actor_id INTEGER NOT NULL, 
	film_id INTEGER NOT NULL, 
	last_update TIMESTAMP NOT NULL, 
	PRIMARY KEY (actor_id, film_id), 
	CONSTRAINT fk_film_actor_actor FOREIGN KEY(actor_id) REFERENCES actor (actor_id) ON UPDATE CASCADE, 
	CONSTRAINT fk_film_actor_film FOREIGN KEY(f

/Users/luizfelipew/Documents/git/AI/Exploring-LLMs/agents/agentes-ai-apps/.venv/lib/python3.13/site-packages/langchain_community/utilities/sql_database.py:313: SAWarning: Cannot correctly sort tables; there are unresolvable cycles between tables "staff, store", which is usually caused by mutually dependent foreign key constraints.  Foreign key constraints involving these tables will not be considered; this warning may raise an error in a future release.
  metadata_table_names = [tbl.name for tbl in self._metadata.sorted_tables]
/Users/luizfelipew/Documents/git/AI/Exploring-LLMs/agents/agentes-ai-apps/.venv/lib/python3.13/site-packages/langchain_community/utilities/sql_database.py:325: SAWarning: Cannot correctly sort tables; there are unresolvable cycles between tables "staff, store", which is usually caused by mutually dependent foreign key constraints.  Foreign key constraints involving these tables will not be considered; this warning may raise an error in a future release.
  for tb


Invoking: `sql_db_query` with `{'query': 'SELECT actor_id, COUNT(actor_id) AS film_count FROM film_actor GROUP BY actor_id ORDER BY film_count DESC LIMIT 10'}`


[(107, 42), (102, 41), (198, 40), (181, 39), (23, 37), (81, 36), (13, 35), (37, 35), (60, 35), (106, 35)]Os atores que atuaram em mais filmes são:

1. Ator com ID 107 atuou em 42 filmes.
2. Ator com ID 102 atuou em 41 filmes.
3. Ator com ID 198 atuou em 40 filmes.
4. Ator com ID 181 atuou em 39 filmes.
5. Ator com ID 23 atuou em 37 filmes.
6. Ator com ID 81 atuou em 36 filmes.
7. Ator com ID 13 atuou em 35 filmes.
8. Ator com ID 37 atuou em 35 filmes.
9. Ator com ID 60 atuou em 35 filmes.
10. Ator com ID 106 atuou em 35 filmes.

> Finished chain.
{'input': 'Quais atores atuaram em mais filmes?', 'output': 'Os atores que atuaram em mais filmes são:\n\n1. Ator com ID 107 atuou em 42 filmes.\n2. Ator com ID 102 atuou em 41 filmes.\n3. Ator com ID 198 atuou em 40 filmes.\n4. Ator com ID 181 atuou em 39 filmes.\n5. Ator com ID 23 

In [15]:
response = agente_executor_sakila.invoke({"input": "Qual é o cliente com mais alugueis?"})
print(response)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


actor, address, category, city, country, customer, film, film_actor, film_category, film_text, inventory, language, payment, rental, staff, store
Invoking: `sql_db_schema` with `{'table_names': 'customer, rental'}`



CREATE TABLE customer (
	customer_id INTEGER NOT NULL, 
	store_id INTEGER NOT NULL, 
	first_name VARCHAR(45) NOT NULL, 
	last_name VARCHAR(45) NOT NULL, 
	email VARCHAR(50) DEFAULT NULL, 
	address_id INTEGER NOT NULL, 
	active CHAR(1) DEFAULT 'Y' NOT NULL, 
	create_date TIMESTAMP NOT NULL, 
	last_update TIMESTAMP NOT NULL, 
	PRIMARY KEY (customer_id), 
	CONSTRAINT fk_customer_store FOREIGN KEY(store_id) REFERENCES store (store_id) ON UPDATE CASCADE, 
	CONSTRAINT fk_customer_address FOREIGN KEY(address_id) REFERENCES address (address_id) ON UPDATE CASCADE
)

/*
3 rows from customer table:
customer_id	store_id	first_name	last_name	email	address_id	active	create_date	last_update
1	1	MARY

/Users/luizfelipew/Documents/git/AI/Exploring-LLMs/agents/agentes-ai-apps/.venv/lib/python3.13/site-packages/langchain_community/utilities/sql_database.py:313: SAWarning: Cannot correctly sort tables; there are unresolvable cycles between tables "staff, store", which is usually caused by mutually dependent foreign key constraints.  Foreign key constraints involving these tables will not be considered; this warning may raise an error in a future release.
  metadata_table_names = [tbl.name for tbl in self._metadata.sorted_tables]
/Users/luizfelipew/Documents/git/AI/Exploring-LLMs/agents/agentes-ai-apps/.venv/lib/python3.13/site-packages/langchain_community/utilities/sql_database.py:325: SAWarning: Cannot correctly sort tables; there are unresolvable cycles between tables "staff, store", which is usually caused by mutually dependent foreign key constraints.  Foreign key constraints involving these tables will not be considered; this warning may raise an error in a future release.
  for tb


Invoking: `sql_db_query` with `{'query': 'SELECT c.first_name, c.last_name, COUNT(r.customer_id) AS num_rentals FROM customer c JOIN rental r ON c.customer_id = r.customer_id GROUP BY c.customer_id ORDER BY num_rentals DESC LIMIT 1'}`


[('ELEANOR', 'HUNT', 46)]O cliente com mais aluguéis é ELEANOR HUNT, com um total de 46 aluguéis.

> Finished chain.
{'input': 'Qual é o cliente com mais alugueis?', 'output': 'O cliente com mais aluguéis é ELEANOR HUNT, com um total de 46 aluguéis.'}
